# 13 · Colab'da Raptor-80 eğitimi (Kaggle GPU kotası dolunca)

**Neden?** Kaggle'da hesap başına haftalık 30 saat GPU var ve bir Raptor-80 eğitimi T4'te ~9.5 saat
sürüyor. Colab Pro'daki A100 / L4 aynı eğitimi ~2–3 saatte bitirir. Bu notebook Kaggle'daki
eğitimin (12) **birebir aynısını** Colab'da çalıştırır ve sonucu Kaggle'a geri yükler.

**Akış:**
1. Kaggle token'ı Colab **Secrets**'tan okunur (sol menü → 🔑 → `KAGGLE_API_TOKEN` adıyla ekleyin,
   "Notebook access" açık olsun). Token koda/çıktıya yazılmaz.
2. Kaggle'dan indirilir: kod + etiketler (`kadirov0/rsna-knee-src`), 80 kesitlik corpus'un 3 parçası
   (~40 GB, `rsna-knee-11-corpus80-s0/1/2`), yarışmanın `train.csv`'si.
3. `third_party/raptor/train_knee.py` aynı argümanlarla çalışır (A100/L4'te otomatik bf16).
4. Checkpoint Kaggle'a **private dataset** olarak yüklenir; final notebook (10) oradan kullanır.

**Gereken:** Runtime → Change runtime type → **A100** (ya da L4). Disk ≥ 100 GB (A100 runtime'da var).

In [ ]:
# ---- Ayarlar ----------------------------------------------------------------------
SEED = 21                      # her Colab eğitimi farklı tohumla (12: 42, 12b: 7, 12c: 13)
LABELS_FILE = "train_labels_v2.csv"
TAG = f"kadirov80_lbl2_s{SEED}"
EPOCHS = 8
BS = 8                         # A100 40/80 GB için; L4 (24 GB) ise 4 yapın
OWNER = "kadirov0"             # kaynakların bulunduğu Kaggle hesabı
# her eğitim kendi private dataset'ine (Kaggle'da yeni sürüm eski dosyaları sildiği için ayrı)
UPLOAD_DATASET = f"{OWNER}/rsna-knee-colab-" + TAG.replace("_", "-")
WORK = "/content/rsna"

In [ ]:
import os, subprocess, sys, json, glob, shutil, time
from pathlib import Path
from google.colab import userdata

os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")   # Secrets'tan; çıktıya yazılmaz
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "kaggle", "timm"], check=True)
Path(WORK).mkdir(parents=True, exist_ok=True)
os.chdir(WORK)
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout)
print(shutil.disk_usage("/content"))

def kg(*args):
    r = subprocess.run([sys.executable, "-m", "kaggle", *args], capture_output=True, text=True)
    print((r.stdout + r.stderr)[-400:])
    if r.returncode:
        raise RuntimeError("kaggle komutu başarısız: " + " ".join(args))

## 1. Girdileri indir (corpus ~40 GB, Colab ağıyla ~10–20 dk)

In [ ]:
t0 = time.time()
kg("datasets", "download", f"{OWNER}/rsna-knee-src", "-p", "src_ds", "--unzip")
for z in glob.glob("src_ds/*.zip"):            # src/ ve third_party/ klasörleri zip olarak gelir
    shutil.unpack_archive(z, "src_ds/" + Path(z).stem); os.remove(z)
kg("competitions", "download", "-c", "rsna-knee-abnormality-detection", "-f", "train.csv", "-p", "inputs")
for z in glob.glob("inputs/*.zip"):
    shutil.unpack_archive(z, "inputs"); os.remove(z)
for i in range(3):
    kg("kernels", "output", f"{OWNER}/rsna-knee-11-corpus80-s{i}", "-p", f"inputs/corpus_s{i}", "--page-size", "200",
       "--file-pattern", r"corpus_.*\.npy$")
print("indirildi:", round(sum(f.stat().st_size for f in Path("inputs").rglob("*") if f.is_file()) / 1e9, 1), "GB",
      f"| {(time.time() - t0) / 60:.1f} dk")
print(sorted(str(p) for p in Path("inputs").rglob("corpus_*")))

## 2. Eğitim (Kaggle'daki 12 ile aynı reçete)

In [ ]:
script = next(Path("src_ds").rglob("train_knee.py"))
labels_csv = next(Path("src_ds").rglob(LABELS_FILE))
import pandas as pd
pd.read_csv(labels_csv).drop(columns=["is_gold"]).to_parquet(f"labels_{TAG}.parquet", index=False)
shutil.copy(script, "train_knee.py")
cmd = [sys.executable, "train_knee.py", "--arch", "coatnet_rmlp_2_rw_384.sw_in12k_ft_in1k", "--res", "384",
       "--epochs", str(EPOCHS), "--bs", str(BS), "--k", "12", "--k_eval", "24", "--grad_ckpt",
       "--workers", "8", "--topk", "3", "--tag", TAG, "--labels", f"labels_{TAG}.parquet",
       "--norm", "imagenet", "--seed", str(SEED), "--log_every", "200"]
env = dict(os.environ, RAPTOR_INPUT_ROOT=str(Path("inputs").resolve()))
proc = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end="")
proc.wait()
assert proc.returncode == 0
print(json.load(open(f"raptor_ft_{TAG}.json"))["best_gold_auc"])

## 3. Sonucu Kaggle'a yükle (private dataset; final notebook buradan okur)

In [ ]:
up = Path("upload"); up.mkdir(exist_ok=True)
for f in glob.glob(f"raptor_ft_{TAG}*.pt") + glob.glob(f"raptor_ft_{TAG}.json") + glob.glob(f"raptor_gold_{TAG}*.npz"):
    shutil.copy(f, up)
meta = {"title": UPLOAD_DATASET.split("/")[1], "id": UPLOAD_DATASET, "licenses": [{"name": "CC0-1.0"}]}
(up / "dataset-metadata.json").write_text(json.dumps(meta))
exists = subprocess.run([sys.executable, "-m", "kaggle", "datasets", "status", UPLOAD_DATASET],
                        capture_output=True, text=True).returncode == 0
if exists:
    kg("datasets", "version", "-p", str(up), "-m", f"add {TAG}")
else:
    kg("datasets", "create", "-p", str(up))
print(sorted(os.listdir(up)))